# 01 — Dataset audit

## Objective
Inspect the three source datasets independently, preserve source files, characterize structure and data quality, and document whether each contains a defensible continuous SER target. No participants are pooled here.

## Reproducibility and limitations
`DATA.xls` is a legacy binary workbook. This notebook converts it to a temporary `.xlsx` with the bundled LibreOffice executable when `xlrd` is unavailable; the original file is never changed. CSV parsing evaluates candidate delimiters and decimal conventions by resulting column structure. A variable name alone is not treated as proof of its measurement definition. The supplied files include no codebook or protocol documentation, so clinical conventions and compatibility remain provisional.

In [1]:
from pathlib import Path
import shutil, subprocess, tempfile, csv, json, re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "DATA.xls").exists():
    ROOT = ROOT.parent if (ROOT.parent / "DATA.xls").exists() else ROOT
OUT = ROOT / "outputs" / "audit"
OUT.mkdir(parents=True, exist_ok=True)
FILES = {"DATA.xls": ROOT / "DATA.xls", "Data3.xlsx": ROOT / "Data3.xlsx", "DataSet_main.csv": ROOT / "DataSet_main.csv"}
for name, path in FILES.items():
    if not path.exists(): raise FileNotFoundError(path)

# CSV delimiter/decimal inference scores well-formed multi-column parses first.
def read_csv_auto(path):
    candidates=[]
    for sep in [",", ";", "\t", "|"]:
        for decimal in [".", ","]:
            try:
                frame=pd.read_csv(path, sep=sep, decimal=decimal, engine="python", skipinitialspace=True)
                score=(frame.shape[1] > 1, frame.shape[1], int(frame.dtypes.apply(lambda d: pd.api.types.is_numeric_dtype(d)).sum()), frame.shape[0], -int(frame.columns.astype(str).str.contains(sep, regex=False).sum()), decimal==".")
                candidates.append((score, sep, decimal, frame))
            except Exception: pass
    if not candidates: raise ValueError("No viable CSV parse")
    _, sep, decimal, frame=max(candidates, key=lambda x:x[0])
    return frame, sep, decimal

# Legacy XLS reader with a non-destructive temporary conversion fallback.
def read_legacy_xls(path):
    try:
        return pd.ExcelFile(path), None
    except ImportError:
        soffice=shutil.which("soffice") or "/Users/omvdangi/.cache/codex-runtimes/codex-primary-runtime/dependencies/bin/override/soffice"
        if not Path(soffice).exists(): raise RuntimeError("Cannot read .xls: install/use xlrd or provide LibreOffice soffice")
        tmp=Path(tempfile.mkdtemp(prefix="myopia_xls_"))
        subprocess.run([soffice,"--headless","--convert-to","xlsx","--outdir",str(tmp),str(path)],check=True,capture_output=True,text=True)
        converted=tmp/(path.stem+".xlsx")
        return pd.ExcelFile(converted), tmp

source_tables={}; inventory=[]; column_rows=[]; unique_rows=[]; invalid_rows=[]; workbook_refs={}
for name,path in FILES.items():
    if name.endswith(".csv"):
        frame, sep, decimal=read_csv_auto(path)
        source_tables[name]={"CSV":frame}
        inventory.append({"dataset":name,"sheet":"CSV","rows":len(frame),"columns":frame.shape[1],"duplicate_rows":int(frame.duplicated().sum()),"delimiter":repr(sep),"decimal":decimal})
    else:
        book,tmp=read_legacy_xls(path) if name.endswith(".xls") else (pd.ExcelFile(path),None)
        workbook_refs[name]=(book,tmp)
        for sheet in book.sheet_names:
            frame=pd.read_excel(book,sheet_name=sheet)
            source_tables[name+"::"+sheet]={"_frame":frame}
            inventory.append({"dataset":name,"sheet":sheet,"rows":len(frame),"columns":frame.shape[1],"duplicate_rows":int(frame.duplicated().sum()),"delimiter":None,"decimal":None})

# Normalize access to frames.
def get_frame(key):
    obj=source_tables[key]
    return obj["CSV"] if "CSV" in obj else obj["_frame"]

for key in source_tables:
    frame=get_frame(key); dataset,sheet=(key.split("::",1)+["-"])[:2] if "::" in key else (key,"CSV")
    for col in frame.columns:
        s=frame[col]; nonnull=s.dropna(); numeric=pd.to_numeric(nonnull.astype(str).str.replace(",",".",regex=False),errors="coerce")
        n_num=int(numeric.notna().sum()); pct=100*n_num/max(1,len(nonnull))
        kind="numerical" if len(nonnull) and pct>=0.9 else ("categorical" if s.nunique(dropna=True)<=30 else "text/object")
        column_rows.append({"dataset":dataset,"sheet":sheet,"column":str(col),"dtype":str(s.dtype),"inferred_type":kind,"n_nonmissing":int(s.notna().sum()),"n_missing":int(s.isna().sum()),"missing_pct":100*s.isna().mean(),"unique_nonmissing":int(s.nunique(dropna=True)),"numeric_parse_pct_nonmissing":pct,"numeric_min":float(numeric.min()) if n_num else np.nan,"numeric_max":float(numeric.max()) if n_num else np.nan})
        values=s.astype(str).value_counts(dropna=False).head(50)
        for value,count in values.items(): unique_rows.append({"dataset":dataset,"sheet":sheet,"column":str(col),"value":value,"count":int(count)})
    # Broad sentinel/string scan only; no sentinel is recoded without a data dictionary.
    for col in frame.columns:
        s=frame[col]; vals=s.astype(str).str.strip().str.lower()
        suspicious=vals.isin(["-1","-9","-99","999","9999","/","na","n/a","unknown","?","."])
        if suspicious.any(): invalid_rows.append({"dataset":dataset,"sheet":sheet,"column":str(col),"candidate_invalid_or_sentinel_count":int(suspicious.sum()),"values":json.dumps(s[suspicious].astype(str).value_counts().to_dict())})

summary=pd.DataFrame(inventory); columns=pd.DataFrame(column_rows); uniques=pd.DataFrame(unique_rows); candidates=pd.DataFrame(invalid_rows)
summary.to_csv(OUT/"dataset_inventory.csv",index=False); columns.to_csv(OUT/"column_profile.csv",index=False); uniques.to_csv(OUT/"top_unique_values.csv",index=False); candidates.to_csv(OUT/"sentinel_candidates.csv",index=False)
display(summary); display(columns.groupby(["dataset","sheet","inferred_type"]).size().rename("columns").reset_index()); display(candidates)

,dataset,sheet,rows,columns,duplicate_rows,delimiter,decimal
0,DATA.xls,2018,3658,11,0,None,None
1,DATA.xls,2014,3057,11,74,None,None
2,DATA.xls,Sheet3,0,0,0,None,None
3,Data3.xlsx,2014-2017,2024,13,15,None,None
4,DataSet_main.csv,CSV,876,23,1,';',","


,dataset,sheet,inferred_type,columns
0,DATA.xls,2014,numerical,11
1,DATA.xls,2018,numerical,11
2,Data3.xlsx,2014-2017,categorical,1
3,Data3.xlsx,2014-2017,numerical,12
4,DataSet_main.csv,CSV,numerical,23


,dataset,sheet,column,candidate_invalid_or_sentinel_count,values
0,DATA.xls,2014,cylindrical refraction of right eye,175,"{""-1"": 171, ""/"": 4}"
1,DATA.xls,2014,axis of right eye,51,"{""/"": 51}"
2,DATA.xls,2014,axial length of right eye,1,"{""/"": 1}"
3,Data3.xlsx,2014-2017,cylinder refraction of right eye,186,"{""-1"": 181, ""/"": 5}"
4,Data3.xlsx,2014-2017,cylinder refraction of left eye,112,"{""-1"": 107, ""/"": 5}"
5,Data3.xlsx,2014-2017,axis of right eye,198,"{""/"": 198}"
6,Data3.xlsx,2014-2017,axis of left eye,198,"{""/"": 198}"
7,Data3.xlsx,2014-2017,AL of right eye,137,"{""/"": 137}"
8,Data3.xlsx,2014-2017,AL of left eye,137,"{""/"": 137}"


## Clinical variable screening
The code below flags likely refraction, axial-length, anthropometric, demographic, environmental, and identifier fields by name for review. These are discovery aids only; names and numerical ranges do not establish definitions. The workbook contains no participant identifier, so repeated participants across study-year sheets cannot be tested from these files alone.

In [2]:
# Explicitly inspect variable-name matches, target candidates, and numeric ranges.
patterns={
 "SER_or_spherical_equivalent":r"ser|spherical.?equivalent|\bse\b",
 "sphere":r"sphere|spherical", "cylinder":r"cyl", "axial_length":r"axial|\bal\b",
 "demographic":r"gender|sex|age|birth", "anthropometric":r"height|weight|bmi",
 "environment_lifestyle":r"study|outdoor|screen|read|sleep|activity|\bpa\b", "identifier":r"(^id$|participant|subject|patient|record)"}
role_rows=[]
for key in source_tables:
    frame=get_frame(key); dataset,sheet=(key.split("::",1)+["-"])[:2] if "::" in key else (key,"CSV")
    for col in frame.columns:
        roles=[role for role,pat in patterns.items() if re.search(pat,str(col),flags=re.I)]
        if roles:
            role_rows.append({"dataset":dataset,"sheet":sheet,"column":str(col),"name_based_candidates":", ".join(roles),"note":"Requires codebook/protocol verification"})
roles=pd.DataFrame(role_rows); roles.to_csv(OUT/"clinical_variable_candidates.csv",index=False); display(roles)

# SER is computed only where explicit sphere and cylinder columns co-occur; convention still requires source documentation.
ser_audit=[]
for key in source_tables:
    f=get_frame(key); cols=list(map(str,f.columns)); dataset,sheet=(key.split("::",1)+["-"])[:2] if "::" in key else (key,"CSV")
    direct=[c for c in cols if re.search(r"(^|\b)(ser|se)(\b|$)|spherical.?equivalent",c,re.I)]
    sph=[c for c in cols if re.search(r"sphere|spherical refraction",c,re.I)]
    cyl=[c for c in cols if re.search(r"cyl",c,re.I)]
    ser_audit.append({"dataset":dataset,"sheet":sheet,"direct_SE_name_candidates":" | ".join(direct),"sphere_candidates":" | ".join(sph),"cylinder_candidates":" | ".join(cyl),"candidate_continuous_SER_possible":bool(sph and cyl and dataset!="DataSet_main.csv"),"interpretation":"Candidate only; requires refraction convention and laterality documentation"})
ser_table=pd.DataFrame(ser_audit); ser_table.to_csv(OUT/"ser_availability.csv",index=False); display(ser_table)

,dataset,sheet,column,name_based_candidates,note
0,DATA.xls,2018,gender,demographic,Requires codebook/protocol verification
1,DATA.xls,2018,time of study at weekdays,environment_lifestyle,Requires codebook/protocol verification
2,DATA.xls,2018,time of study at weekend days,environment_lifestyle,Requires codebook/protocol verification
3,DATA.xls,2018,time outdoors at weekdays,environment_lifestyle,Requires codebook/protocol verification
4,DATA.xls,2018,time outdoors at weekend days,environment_lifestyle,Requires codebook/protocol verification
5,DATA.xls,2018,height,anthropometric,Requires codebook/protocol verification
6,DATA.xls,2018,weight,anthropometric,Requires codebook/protocol verification
7,DATA.xls,2018,spherical refraction of right eye,sphere,Requires codebook/protocol verification
8,DATA.xls,2018,cylindrical refraction of right eye,cylinder,Requires codebook/protocol verification
9,DATA.xls,2018,axial length of right eye,axial_length,Requires codebook/protocol verification


,dataset,sheet,direct_SE_name_candidates,sphere_candidates,cylinder_candidates,candidate_continuous_SER_possible,interpretation
0,DATA.xls,2018,,spherical refraction of right eye,cylindrical refraction of right eye,True,Candidate only; requires refraction convention...
1,DATA.xls,2014,,spherical refraction of right eye,cylindrical refraction of right eye,True,Candidate only; requires refraction convention...
2,DATA.xls,Sheet3,,,,False,Candidate only; requires refraction convention...
3,Data3.xlsx,2014-2017,,spherical refraction of right eye | spherical ...,cylinder refraction of right eye | cylinder re...,True,Candidate only; requires refraction convention...
4,DataSet_main.csv,CSV,SE,,,False,Candidate only; requires refraction convention...


## Initial dataset compatibility assessment

- **`DATA.xls` 2018 and 2014 sheets:** Both contain same-named demographic, lifestyle, refraction, and axial-length fields; the 2014 sheet has inconsistent object typing and missing axis values. The files provide no participant IDs or protocol definitions, so repeated participants and longitudinal independence cannot be checked. Treat study periods separately pending documentation.
- **`Data3.xlsx`:** Contains birth year, examination year, bilateral sphere/cylinder/axial-length candidates, and anthropometrics. It may support an external-validation design only after measurement protocol, refraction convention, age derivation, and cohort independence are verified. No matching IDs to `DATA.xls` are present.
- **`DataSet_main.csv`:** Semicolon-delimited with comma-decimal `SE` text and 876 rows. It has encoded categorical variables and a candidate `SE` field, but lacks a codebook, measurement definition, laterality, units, and derivation. `SE` cannot be accepted as continuous SER yet. Keep as an independent, undocumented cohort.
- **Pooling:** No combination is currently defensible. Similar variable labels do not establish compatible measurement, recruitment, or participant overlap.

### Audit limitations and next steps
There is no codebook or clinical protocol supplied. Sentinel candidates are reported but not recoded; range checks are only screening and do not determine validity. Before modeling, verify the clinical convention for cylinder sign and axis/refraction, confirm how the 2018/2014 sheets were collected, and obtain definitions for all encoded CSV variables and `SE`.